In [108]:
# Exceptions

class ImageEncoderError(Exception):
    """Custom exception for errors related to image encoding."""
    def __init__(self, message: str):
        super().__init__(message)

class MessageBuilderError(Exception):
    """Custom exception for errors related to message building."""
    def __init__(self, message: str):
        super().__init__(message)

class APIClientError(Exception):
    """Custom exception for errors related to OpenRouter client operations."""
    def __init__(self, message: str):
        super().__init__(message)

In [ ]:
# Message API

import numpy as np
import cv2
import os
import base64
import json

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Dict, List, Optional, Union, Any
from enum import Enum, auto
from pathlib import Path
from openai import OpenAI

class MessageRole(Enum):
    SYSTEM = "system"
    DEVELOPER = "developer"
    USER = "user"
    ASSISTANT = "assistant"

class ContentType(Enum):
    TEXT = auto()
    IMAGE = auto()

@dataclass(frozen=True, slots=True)
class MessageContent:
    type: ContentType
    data: str

@dataclass(frozen=True, slots=True)
class Message:
    role : MessageRole
    content : Union[str, List[MessageContent]]

class ImageEncoder:
    @staticmethod
    def encode_image(image: Union[Path, np.ndarray]) -> str:
        if isinstance(image, Path):
            try:
                if not image.exists():
                    raise ImageEncoderError(f"Image file {image} does not exist.")

                if image.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
                    raise ImageEncoderError(f"Unsupported image format: {image.suffix}. Supported formats are .jpg, .jpeg, and .png.")

                with open(image, "rb") as img_file:
                    b64_image = base64.b64encode(img_file.read()).decode('utf-8')
                    b64_image_str = f"data:image/{image.suffix[1:]};base64,{b64_image}"

            except IOError as e:
                raise ImageEncoderError(f"Error reading image file {image}: {e}")

        elif isinstance(image, np.ndarray):
            if image.ndim != 3 or image.shape[2] not in [3, 4]:
                raise ImageEncoderError("Invalid image array shape. Expected a 3D array with 3 (RGB) or 4 (RGBA) channels.")

            _, buffer = cv2.imencode('.png', image)
            b64_image = base64.b64encode(buffer).decode('utf-8')
            b64_image_str = f"data:image/png;base64,{b64_image}"

        else:
            raise TypeError("Image must be a numpy array or a Path object pointing to an image file.")

        return b64_image_str

class ContentBuilder:
    @staticmethod
    def create_text_content(text : str) -> MessageContent:
        return MessageContent(type=ContentType.TEXT, data=text)

    @staticmethod
    def create_image_content(image: Union[Path, np.ndarray]) -> MessageContent:
        b64_image_str = ImageEncoder.encode_image(image)
        return MessageContent(type=ContentType.IMAGE, data=b64_image_str)

class MessageBuilder:
    def __init__(self, role : MessageRole):
        self._role = role
        self._content: List[MessageContent] = []

    def add_text_content(self, text: str):
        self._content.append(ContentBuilder.create_text_content(text))

    def add_image_content(self, image: Union[Path, np.ndarray]):
        self._content.append(ContentBuilder.create_image_content(image))

    def build_message(self) -> Message:
        message = None

        if not self._role:
            raise MessageBuilderError("Message role cannot be empty")

        if not self._content:
            raise MessageBuilderError("Message content cannot be empty")

        if len(self._content) == 1:
            # If there's only one content item, return it directly
            message = Message(role=self._role, content=self._content[0].data)
        else:
            # If there are multiple content items, return them as a list
            message = Message(role=self._role, content=self._content)

        return message

In [ ]:
# API Clients

class APIClient(ABC):
    @abstractmethod
    def send_message(self, messages: List[Message], **kwargs) -> str:
        """Send a message to the API and return the response."""
        pass

    @abstractmethod
    def _format_messages(self, messages: List[Message]) -> List[Dict[str, str]]:
        """Format messages for the API request."""
        pass

    @abstractmethod
    def _format_content(self, content: MessageContent) -> Dict[str, str]:
        """Format content for the API request."""
        pass

class OpenAIClient(APIClient):
    def __init__(self, model_name: str):
        self.api_key = os.getenv("OPENAI_API_KEY")
        if not self.api_key:
            raise APIClientError("OPENAI_API_KEY environment variable is not set.")

        self.client = OpenAI(api_key=self.api_key)
        self.model_name = model_name

    def _format_messages(self, messages: List[Message]) -> List[Dict[str, str]]:
        formatted_messages = []
        for message in messages:
            formatted_message = {
                "role": message.role.value,
                "content": message.content if isinstance(message.content, str) else [self._format_content(content) for content in message.content]
            }
            formatted_messages.append(formatted_message)
        return formatted_messages

    def _format_content(self, content: MessageContent) -> Dict[str, str]:
        if content.type == ContentType.TEXT:
            return {"type": "input_text", "text": content.data}
        elif content.type == ContentType.IMAGE:
            return {"type": "input_image", "image_url": content.data}
        else:
            raise APIClientError(f"Unsupported content type: {content.type}")

    def send_message(self, messages: List[Message], **kwargs) -> str:
        if not messages:
            raise APIClientError("Messages cannot be empty.")

        formatted_messages = self._format_messages(messages)
        response = self.client.responses.create(
            model="gpt-4.1-mini-2025-04-14",
            input=formatted_messages,
            temperature=0.1,
            max_output_tokens=500,
        )

        return response.output_text

class OpenRouterClient(APIClient):
    BASE_URL = "https://openrouter.ai/api/v1"
    def __init__(self, model_name: str):
        self.api_key = os.getenv("OPENROUTER_API_KEY")
        if not self.api_key:
            raise APIClientError("OPENROUTER_API_KEY environment variable is not set.")

        self.client = OpenAI(api_key=self.api_key, base_url=OpenRouterClient.BASE_URL)
        self.model_name = model_name

    def _format_messages(self, messages: List[Message]) -> List[Dict[str, str]]:
        formatted_messages = []
        for message in messages:
            formatted_message = {
                "role": message.role.value,
                "content": message.content if isinstance(message.content, str) else [self._format_content(content) for content in message.content]
            }
            formatted_messages.append(formatted_message)
        return formatted_messages


    def _format_content(self, content: MessageContent) -> Dict[str, str]:
        if content.type == ContentType.TEXT:
            return {"type": "text", "text": content.data}
        elif content.type == ContentType.IMAGE:
            return {
                "type": "image_url",
                "image_url": {
                    "url" : content.data
                }
            }
        else:
            raise APIClientError(f"Unsupported content type: {content.type}")

    def send_message(self, messages: List[Message], **kwargs) -> str:
        if not messages:
            raise APIClientError("Messages cannot be empty.")

        formatted_messages = self._format_messages(messages)

        response = self.client.chat.completions.create(
            model=self.model_name,
            extra_body={},
            messages=formatted_messages,
            temperature=0.1,
            max_completion_tokens=250,
        )

        return response.choices[0].message.content

In [ ]:
# Model Registry and Instantiator
import base64
import os
import pathlib
from dataclasses import dataclass, asdict
from enum import Enum, auto
from typing import Any, Iterable, List, Mapping, Protocol, Sequence

class ModelCatalogue:
    _catalogue : Mapping[str, Sequence[str]] = {
        "openai": ["openai/gpt-4.1-2025-04-14"],
        "qwen": ["qwen/qwen2.5-vl-72b-instruct:free"],
        "google": ["google/gemini-2.5-flash"],
        "anthropic": ["anthropic/claude-3-7-sonnet-20250219"],
    }

    @classmethod
    def validate(cls, model_name: str):
        provider = model_name.split("/", 1)[0]
        if provider not in cls._catalogue:
            raise ValueError(
                f"Unknown provider '{provider}'. Valid providers: {list(cls._catalogue)}"
            )
        if model_name not in cls._catalogue[provider]:
            raise ValueError(
                f"Unknown model '{model_name}' for provider '{provider}'. "
                f"Choices: {cls._catalogue[provider]}"
            )

    @classmethod
    def providers(cls) -> Sequence[str]:
        return tuple(cls._catalogue)

    @classmethod
    def models(cls, provider: str) -> Sequence[str]:
        return tuple(cls._catalogue[provider])

class VLMAgent:
    def __init__(self, model_name: str, **kwargs: Any):
        ModelCatalogue.validate(model_name)

        self.model_name = model_name
        self.client = OpenRouterClient(self.model_name)
        # self.client = OpenAIClient(self.model_name)

        self.temperature = kwargs.get("temperature", 0.1)
        self.max_output_tokens = kwargs.get("max_output_tokens", 500)

    def create_user_message(self, text : str = None, image : Union[Path, np.ndarray] = None) -> Message:
        builder = MessageBuilder(MessageRole.USER)

        if text:
            builder.add_text_content(text)

        if image:
            builder.add_image_content(image)

        return builder.build_message()

    def create_system_message(self, text: str) -> Message:
        builder = MessageBuilder(MessageRole.DEVELOPER)
        # builder = MessageBuilder(MessageRole.SYSTEM)
        builder.add_text_content(text)
        return builder.build_message()

    def send_message(self, messages: List[Message]) -> str:
        if not messages:
            raise ValueError("Messages cannot be empty.")

        return self.client.send_message(messages, temperature=self.temperature, max_output_tokens=self.max_output_tokens)

In [ ]:
class SceneAnalyzer(VLMAgent):
    def __init__(
            self,
            model_name: str = "qwen/qwen2.5-vl-72b-instruct:free",
            **kwargs):
        super().__init__(model_name, kwargs=kwargs)

    def interpret_scene(self, scene_context_path: str, image_path: Path):
        system_instruction = f"""
You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
representation of the scene metadata. Given this information, provide a concise, natural language summary of
the current driving context.

Also, identify relevant actors in the scene based on their pose and proximity to
the ego vehicle. Note that leading vehicles are more important than trailing vehicles in the ego lane, whereas
both are important in adjacent lanes. In the textual summary, lane adjacency is provided as (DIRECTION - n) where
DIRECTION is either "Left" or "Right" and n is the lane number relative to the ego vehicle's lane.

Your response should be structured as follows:

road_description: Description of the road network (junction, highway, local road, etc.).
traffic_description: Description of the traffic conditions in the scene.
static_objects_and_obstacles_description: Description of static objects and obstacles in the scene (if present).
ego_vehicle_description: Description of the state of the ego vehicle.
key_actors: Description of nearby actors with their IDs, poses, and states that are relevant to the driving context.
reasoning: Brief description justifying your choice of key actors.
        """

        with open(scene_context_path, 'r') as file:
            scene_context = file.read()

        system_message = self.create_system_message(text=system_instruction)
        user_message = self.create_user_message(text=scene_context, image=image_path)

        messages = [system_message, user_message]

        response = self.send_message(messages)

        return response

    def predict_intentions(self, scene_context_path: str, image_path: Path, key_actor_description: str):
        system_instruction = f"""
You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
representation of the scene metadata and a corresponding natural language summary. This includes information
about key actors that may be relevant to the ego vehicle's immediate decision-making process.

Given this information, predict the likely intentions of the key actors in the scene considering ego vehicle's
remaining route which is shown as a green line in the image. Note that NPC actors are simplistic and do not
account for the ego vehicle's state when making decisions. Thus, opt for NPC actor predictions that allow
the ego to prioritize its own safety and make conservative decisions.

For all actors, consider their current pose, speed, and proximity to the ego vehicle. For vehicles, their set of
possible actions are: "follow_lane", "change_lane_left", "change_lane_right", "turn_left", "turn_right",
"stop". For pedestrians, their actions are: "cross_street", "wait", and "walk".

Your response should be structured as follows:

key_actor_id: ID of the key actor.
key_actor_type: Type of the key actor (e.g., vehicle, pedestrian).
key_actor_intention: Predicted intention of the key actor based on the scene context.
key_actor_reasoning: Brief description justifying the predicted intention of the key actor.
        """

        with open(scene_context_path, 'r') as file:
            scene_context = file.read()

        system_message = self.create_system_message(text=system_instruction)

        separator_text = "\n\nKey Actor Description:\n"
        scene_context_with_key_actor = scene_context + separator_text + key_actor_description
        user_message = self.create_user_message(text=scene_context_with_key_actor, image=image_path)

        messages = [system_message, user_message]

        response = self.send_message(messages)

        return response

    def plan_ego_actions(self, key_actor_intentions: str, image_path: Path):
        system_instruction = f"""
You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
summary of likely intentions of key actors in the scene.

Given this information, generate a sequence of high-level driving actions for the ego vehicle over a 3s planning
horizon. The plan should prioritize the ego vehicle's safety and conservative decision-making, taking into account the
predicted intentions of the key actors.

Here's the set of possible actions for the ego vehicle:
- `accelerate`: Accelerate the ego vehicle
- `decelerate`: Decelerate the ego vehicle
- `maintain_speed`: Maintain the ego vehicle's current speed
- `change_lane_left`: Change the ego vehicle's lane to the left.
- `change_lane_right`: Change the ego vehicle's lane to the right.

Your response should be structured as follows:
plan: A sequence of high-level driving actions for the ego vehicle over a 3s planning horizon. Each action should be
    separated by a comma.
reasoning: Brief description justifying the chosen actions for the ego vehicle.
        """

        system_message = self.create_system_message(text=system_instruction)
        user_message = self.create_user_message(text=key_actor_intentions, image=image_path)

        messages = [system_message, user_message]

        response = self.send_message(messages)

        return response

In [ ]:
import os


In [114]:
scene_analyzer = SceneAnalyzer()

image_path = Path("/home/carla/carla_garage/leaderboard_autopilot/runs/run_20250702_001202/rgb_bounding_boxes_0079.png")
text_path = "/home/carla/carla_garage/leaderboard_autopilot/runs/run_20250702_001202/scene_context_0079.txt"

scene_description = scene_analyzer.interpret_scene(scene_context_path=text_path, image_path=image_path)
print("Scene Description:")
print(scene_description)

actor_intentions = scene_analyzer.predict_intentions(scene_context_path=text_path, image_path=image_path, key_actor_description=scene_description)
print("\nActor Intentions:")
print(actor_intentions)

ego_actions = scene_analyzer.plan_ego_actions(key_actor_intentions=actor_intentions, image_path=image_path)
print("\nEgo Vehicle Actions:")
print(ego_actions)

Scene Description:
road_description: The scene takes place on a multi-lane highway with clear lane markings. The road is flanked by trees and barriers, indicating a controlled traffic environment.

traffic_description: Traffic is moderate, with vehicles present in adjacent lanes. The ego vehicle is traveling at a moderate speed of approximately 16.1 m/s, while the surrounding vehicles are moving at varying speeds.

static_objects_and_obstacles_description: There are no significant static objects or obstacles in the immediate vicinity of the ego vehicle. The road is clear of any barriers or debris that could impede movement.

ego_vehicle_description: The ego vehicle is traveling in its lane at a steady speed of 16.1 m/s. It is not planning any lane changes and is maintaining its current trajectory.

key_actors: 
- Vehicle ID: 3696, in the Right-1 lane, is a leading vehicle with a relative distance of 10.82 meters. It is traveling at a speed of 11.8 m/s and is slightly ahead of the ego v

In [ ]:
import os
import numpy as np
import base64

from PIL import Image
from openai import OpenAI

class OpenRouterAgent:
    model_names = {
        "openai" : ["openai/gpt-4.1-2025-04-14"],
        "qwen" : ["qwen/qwen2.5-vl-72b-instruct:free"],
        "google" : ["google/gemini-2.5-flash"],
        "anthropic" : ["anthropic/claude-3-7-sonnet-20250219"]
    }

    def __init__(self, model_name: str = "openai/gpt-4.1-2025-04-14"):
        model_provider = model_name.split('/')[0]
        if model_provider not in self.model_names:
            raise ValueError(f"Invalid model provider: {model_provider}. Available providers: {list(self.model_names.keys())}")

        self.model_name = model_name
        if self.model_name not in self.model_names[model_provider]:
            raise ValueError(f"Invalid model name: {self.model_name}. Available models for {model_provider}: {self.model_names[model_provider]}")

        self.api_key = os.getenv("OPENROUTER_API_KEY")
        if not self.api_key:
            raise ValueError("OPENROUTER_API_KEY environment variable is not set.")

        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=self.api_key
        )

class VLMAgent(OpenRouterAgent):
    def __init__(self, temperature: float = 0.1, max_output_tokens: int = 500, model_name: str = "openai/gpt-4.1-2025-04-14"):
        super().__init__(model_name=model_name)

        self.temperature = temperature
        self.max_output_tokens = max_output_tokens

    def __create_message(self, role: str, content: str):
        return {
            "role": role,
            "content": content
        }

    def __create_message_content(self, type: str, text_content: str = None, image_path: str = None):
        def encode_image(image_path: str):
            with open(image_path, "rb") as image_file:
                return base64.b64encode(image_file.read()).decode('utf-8')

        content = {}
        if type == "input_text":
            if not text_content:
                raise ValueError("Text content must be provided for input_text type.")
            content = {
                "type" : type,
                "text" : text_content
            }
        elif type == "input_image":
            if not image_path:
                raise ValueError("Image path must be provided for input_image type.")
            base64_image_data = encode_image(image_path)
            content = {
                "type"          : type,
                "image_url"     : f"data:image/jpeg;base64,{base64_image_data}",
            }
        return content

    def __create_text_message_content(self, text: str):
        text_content = self.__create_message_content(type="input_text", text_content=text)

        return text_content

    def __create_image_message_content(self, image_path: str):
        image_content = self.__create_message_content(type="input_image", image_path=image_path)

        return image_content

    def _create_user_message(self, text: str = None, image_path: str = None):
        user_message_content = []

        if text:
            user_message_content.append(self.__create_text_message_content(text=text))
        if image_path:
            user_message_content.append(self.__create_image_message_content(image_path=image_path))

        user_message = self.__create_message(role="user", content=user_message_content)

        return user_message

    def _create_system_message(self, text: str):
        message = self.__create_message(role="developer", content=text)

        return message

class SceneAnalyzer(VLMAgent):
    def __init__(
            self,
            temperature: float = 0.1,
            max_output_tokens: int = 500,
            model_name: str = "gpt-4.1-2025-04-14"):
        super().__init__(temperature, max_output_tokens, model_name)

    def interpret_scene(self, scene_context_path: str, image_path: str):
        system_instruction = f"""
You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
representation of the scene metadata. Given this information, provide a concise, natural language summary of
the current driving context.

Also, identify relevant actors in the scene based on their pose and proximity to
the ego vehicle. Note that leading vehicles are more important than trailing vehicles in the ego lane, whereas
both are important in adjacent lanes. In the textual summary, lane adjacency is provided as (DIRECTION - n) where
DIRECTION is either "Left" or "Right" and n is the lane number relative to the ego vehicle's lane.

Your response should be structured as follows:

road_description: Description of the road network (junction, highway, local road, etc.).
traffic_description: Description of the traffic conditions in the scene.
static_objects_and_obstacles_description: Description of static objects and obstacles in the scene (if present).
ego_vehicle_description: Description of the state of the ego vehicle.
key_actors: Description of nearby actors with their IDs, poses, and states that are relevant to the driving context.
reasoning: Brief description justifying your choice of key actors.
        """

        with open(scene_context_path, 'r') as file:
            scene_context = file.read()

        system_message = self._create_system_message(text=system_instruction)
        user_message = self._create_user_message(text=scene_context, image_path=image_path)

        messages = [system_message, user_message]

        response = self.client.responses.create(
            model=self.model_name,
            input=messages,
            temperature=self.temperature,
            max_output_tokens=self.max_output_tokens,
        )

        return response.output_text

    def predict_intentions(self, scene_context_path: str, image_path: str, key_actor_description: str):
        system_instruction = f"""
You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
representation of the scene metadata and a corresponding natural language summary. This includes information
about key actors that may be relevant to the ego vehicle's immediate decision-making process.

Given this information, predict the likely intentions of the key actors in the scene considering ego vehicle's
remaining route which is shown as a green line in the image. Opt for predictions that prioritize the ego's safety
and conservative decision-making.

For all actors, consider their current pose, speed, and proximity to the ego vehicle. For vehicles, their set of
possible actions are: "follow_lane", "change_lane_left", "change_lane_right", "turn_left", "turn_right",
"stop", "accelerate", and "decelerate". For pedestrians, their actions are: "cross_street", "wait", and "walk".

Your response should be structured as follows:

key_actor_id: ID of the key actor.
key_actor_type: Type of the key actor (e.g., vehicle, pedestrian).
key_actor_intention: Predicted intention of the key actor based on the scene context.
key_actor_reasoning: Brief description justifying the predicted intention of the key actor.
        """

        with open(scene_context_path, 'r') as file:
            scene_context = file.read()

        system_message = self._create_system_message(text=system_instruction)

        separator_text = "\n\nKey Actor Description:\n"
        scene_context_with_key_actor = scene_context + separator_text + key_actor_description
        user_message = self._create_user_message(text=scene_context_with_key_actor, image_path=image_path)

        messages = [system_message, user_message]

        response = self.client.responses.create(
            model=self.model_name,
            input=messages,
            temperature=self.temperature,
            max_output_tokens=self.max_output_tokens,
        )

        return response.output_text

    def plan_ego_actions(self, key_actor_intentions: str, image_path: str):
        system_instruction = f"""
You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
summary of likely intentions of key actors in the scene.

Given this information, generate a sequence of high-level driving actions for the ego vehicle over a 3s planning
horizon. The plan should prioritize the ego vehicle's safety and conservative decision-making, taking into account the
predicted intentions of the key actors.

Here's the set of possible actions for the ego vehicle:
- `accelerate`: Accelerate the ego vehicle
- `decelerate`: Decelerate the ego vehicle
- `maintain_speed`: Maintain the ego vehicle's current speed
- `change_lane_left`: Change the ego vehicle's lane to the left.
- `change_lane_right`: Change the ego vehicle's lane to the right.

Your response should be structured as follows:
plan: A sequence of high-level driving actions for the ego vehicle over a 3s planning horizon. Each action should be
    separated by a comma.
reasoning: Brief description justifying the chosen actions for the ego vehicle.
        """

        system_message = self._create_system_message(text=system_instruction)
        user_message = self._create_user_message(text=key_actor_intentions, image_path=image_path)

        messages = [system_message, user_message]

        response = self.client.responses.create(
            model=self.model_name,
            input=messages,
            temperature=self.temperature,
            max_output_tokens=self.max_output_tokens,
        )

        return response.output_text

In [23]:
scene_analyzer = SceneAnalyzer()

image_path = "/home/carla/carla_garage/leaderboard_autopilot/runs/run_20250702_001202/rgb_bounding_boxes_0079.png"
text_path = "/home/carla/carla_garage/leaderboard_autopilot/runs/run_20250702_001202/scene_context_0079.txt"

scene_description = scene_analyzer.interpret_scene(scene_context_path=text_path, image_path=image_path)
print("Scene Description:")
print(scene_description)

actor_intentions = scene_analyzer.predict_intentions(scene_context_path=text_path, image_path=image_path, key_actor_description=scene_description)
print("\nActor Intentions:")
print(actor_intentions)

ego_actions = scene_analyzer.plan_ego_actions(key_actor_intentions=actor_intentions, image_path=image_path)
print("\nEgo Vehicle Actions:")
print(ego_actions)

TypeError: Image must be a numpy array or a Path object pointing to an image file.